In [3]:
# ============================================================
# Jester_03_LightGCN.ipynb
# Part 1 — Imports + Load Data
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Folder path
JESTER_DIR = Path.home() / "Downloads" / "Joke_Project_Diss2026" / "Jester"

# Load edges + jokes (same as Notebook 2)
edges = pd.read_csv(JESTER_DIR / "jester_edges_long.csv")
jokes_df = pd.read_csv(JESTER_DIR / "jester_jokes.csv")

print("edges shape:", edges.shape)
print("jokes_df shape:", jokes_df.shape)

# Make implicit likes (same rule as TF-IDF baseline)
edges["label"] = (edges["rating"] > 0).astype(int)
pos = edges[edges["label"] == 1][["user_id", "joke_id"]].copy()

# Filter sparse users (same as Notebook 2)
MIN_LIKES_PER_USER = 10
user_like_counts = pos["user_id"].value_counts()
keep_users = user_like_counts[user_like_counts >= MIN_LIKES_PER_USER].index
pos = pos[pos["user_id"].isin(keep_users)].reset_index(drop=True)

print("Filtered positives:", len(pos))
print("Users:", pos["user_id"].nunique(), "| Items:", pos["joke_id"].nunique())


edges shape: (1810455, 3)
jokes_df shape: (100, 2)
Filtered positives: 1073324
Users: 24271 | Items: 100


In [4]:
# ============================================================
# Part 2 — Train/Test split per user (same seed + same logic)
# ============================================================

rng = np.random.default_rng(42)
TEST_PER_USER = 2

train_rows, test_rows = [], []

for user_id, grp in pos.groupby("user_id"):
    jokes = grp["joke_id"].to_numpy()
    if len(jokes) <= TEST_PER_USER:
        continue

    test_jokes = rng.choice(jokes, size=TEST_PER_USER, replace=False)
    train_jokes = np.setdiff1d(jokes, test_jokes)

    train_rows.extend([(user_id, j) for j in train_jokes])
    test_rows.extend([(user_id, j) for j in test_jokes])

train_pos = pd.DataFrame(train_rows, columns=["user_id", "joke_id"])
test_pos  = pd.DataFrame(test_rows,  columns=["user_id", "joke_id"])

print("Train positives:", len(train_pos))
print("Test positives:", len(test_pos))
print("Users in test:", test_pos["user_id"].nunique())


Train positives: 1024782
Test positives: 48542
Users in test: 24271


In [5]:
# ============================================================
# Part 3 — Remap IDs to contiguous indices
# ============================================================

unique_users = np.sort(train_pos["user_id"].unique())
unique_items = np.sort(train_pos["joke_id"].unique())

user2idx = {u: i for i, u in enumerate(unique_users)}
item2idx = {j: i for i, j in enumerate(unique_items)}
idx2item = {i: j for j, i in item2idx.items()}  # for printing later

num_users = len(unique_users)
num_items = len(unique_items)

print("num_users:", num_users, "| num_items:", num_items)

train_u = train_pos["user_id"].map(user2idx).to_numpy()
train_i = train_pos["joke_id"].map(item2idx).to_numpy()

test_u = test_pos["user_id"].map(user2idx).to_numpy()
test_i = test_pos["joke_id"].map(item2idx).to_numpy()


num_users: 24271 | num_items: 100


In [6]:
# ============================================================
# Part 4 — Build normalized adjacency for LightGCN
# ============================================================

# Graph has (num_users + num_items) nodes
num_nodes = num_users + num_items

# Build undirected edges: user<->item
u_nodes = train_u
i_nodes = train_i + num_users  # shift items into second block

row = np.concatenate([u_nodes, i_nodes])
col = np.concatenate([i_nodes, u_nodes])

# Edge weights initially 1
data = np.ones(len(row), dtype=np.float32)

# Build sparse adjacency in torch
indices = torch.tensor(np.vstack([row, col]), dtype=torch.long)
values  = torch.tensor(data, dtype=torch.float32)
A = torch.sparse_coo_tensor(indices, values, (num_nodes, num_nodes)).coalesce()

# Compute D^-0.5 * A * D^-0.5 normalization
deg = torch.sparse.sum(A, dim=1).to_dense()
deg_inv_sqrt = torch.pow(deg, -0.5)
deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0

r, c = A.indices()
norm_vals = deg_inv_sqrt[r] * A.values() * deg_inv_sqrt[c]
A_norm = torch.sparse_coo_tensor(A.indices(), norm_vals, A.shape).coalesce()

print("A_norm built with edges:", A_norm._nnz())


A_norm built with edges: 2049564


In [7]:
# ============================================================
# Part 5 — LightGCN model + training utilities
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class LightGCN(nn.Module):
    """LightGCN with trainable ID embeddings (no node features)."""
    def __init__(self, num_users, num_items, embed_dim=64, num_layers=2):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embed_dim = embed_dim
        self.num_layers = num_layers

        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def propagate(self, A_norm):
        """Compute final embeddings via layer-wise graph propagation."""
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        embs = [all_emb]

        for _ in range(self.num_layers):
            all_emb = torch.sparse.mm(A_norm, all_emb)
            embs.append(all_emb)

        # Mean of embeddings across layers (LightGCN)
        out = torch.stack(embs, dim=0).mean(dim=0)
        users_final = out[:self.num_users]
        items_final = out[self.num_users:]
        return users_final, items_final

def bpr_loss(u_emb, pos_emb, neg_emb):
    """Bayesian Personalized Ranking loss."""
    pos_scores = (u_emb * pos_emb).sum(dim=1)
    neg_scores = (u_emb * neg_emb).sum(dim=1)
    return -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

# Build user->set(items) for negative sampling
train_user_items = {}
for u, it in zip(train_u, train_i):
    train_user_items.setdefault(int(u), set()).add(int(it))

all_item_idx = np.arange(num_items)

def sample_negatives(users_np):
    """Sample one negative item per user (not in their train likes)."""
    negs = []
    for u in users_np:
        seen = train_user_items[int(u)]
        while True:
            j = int(np.random.choice(all_item_idx))
            if j not in seen:
                negs.append(j)
                break
    return np.array(negs, dtype=np.int64)


Device: cpu


In [8]:
# ============================================================
# Part 6 — Train LightGCN (BPR) [FIXED: propagate per batch]
# ============================================================

model = LightGCN(num_users, num_items, embed_dim=64, num_layers=2).to(device)
A_norm = A_norm.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

BATCH_SIZE = 4096
EPOCHS = 10
np.random.seed(42)

train_pairs = np.vstack([train_u, train_i]).T  # (N, 2)

for epoch in range(1, EPOCHS + 1):
    model.train()

    # Shuffle edges each epoch
    perm = np.random.permutation(len(train_pairs))
    shuffled = train_pairs[perm]

    losses = []

    for start in range(0, len(shuffled), BATCH_SIZE):
        batch = shuffled[start:start + BATCH_SIZE]
        bu = batch[:, 0]
        bi = batch[:, 1]
        bj = sample_negatives(bu)

        bu_t = torch.tensor(bu, dtype=torch.long, device=device)
        bi_t = torch.tensor(bi, dtype=torch.long, device=device)
        bj_t = torch.tensor(bj, dtype=torch.long, device=device)

        # ✅ Propagate inside the batch so each backward has its own graph
        users_final, items_final = model.propagate(A_norm)

        loss = bpr_loss(users_final[bu_t], items_final[bi_t], items_final[bj_t])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    print(f"Epoch {epoch:02d} | BPR loss: {np.mean(losses):.4f}")


Epoch 01 | BPR loss: 0.6923
Epoch 02 | BPR loss: 0.6756
Epoch 03 | BPR loss: 0.6471
Epoch 04 | BPR loss: 0.6403
Epoch 05 | BPR loss: 0.6394
Epoch 06 | BPR loss: 0.6395
Epoch 07 | BPR loss: 0.6397
Epoch 08 | BPR loss: 0.6391
Epoch 09 | BPR loss: 0.6394
Epoch 10 | BPR loss: 0.6394


In [9]:
# ============================================================
# Fix — Rebuild test_u/test_i (if kernel restarted / vars missing)
# ============================================================

# Remap test user/item IDs using the mappings from Part 4
# (user2idx and item2idx must exist; if not, rerun Part 4 first)

if "user2idx" not in globals() or "item2idx" not in globals():
    raise NameError("user2idx/item2idx not found. Run Part 4 (Remap IDs) first.")

test_u = test_pos["user_id"].map(user2idx).to_numpy()
test_i = test_pos["joke_id"].map(item2idx).to_numpy()

print("test_u/test_i rebuilt.")
print("test_u length:", len(test_u), "| test_i length:", len(test_i))


test_u/test_i rebuilt.
test_u length: 48542 | test_i length: 48542


In [10]:
# ============================================================
# Part 7 — Evaluate LightGCN (Precision/Recall/NDCG)
# ============================================================

import numpy as np
import torch

def ndcg_at_k(recommended_ids, true_ids, k):
    recommended_ids = recommended_ids[:k]
    dcg = 0.0
    for i, jid in enumerate(recommended_ids):
        rel = 1.0 if jid in true_ids else 0.0
        dcg += rel / np.log2(i + 2)

    ideal_hits = min(len(true_ids), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

test_truth = {}
for u, it in zip(test_u, test_i):
    test_truth.setdefault(int(u), set()).add(int(it))

@torch.no_grad()
def evaluate_lightgcn(k=10, max_users=2000):
    model.eval()
    users_final, items_final = model.propagate(A_norm)

    users = np.array(list(test_truth.keys()), dtype=np.int64)[:max_users]

    precisions, recalls, ndcgs = [], [], []

    for u in users:
        truth = test_truth[int(u)]
        seen = train_user_items[int(u)]

        scores = torch.matmul(users_final[u].unsqueeze(0), items_final.T).squeeze(0)

        if len(seen) > 0:
            scores[list(seen)] = -1e9

        topk = torch.topk(scores, k=k).indices.cpu().numpy().tolist()

        hits = sum(1 for it in topk if it in truth)
        precisions.append(hits / k)
        recalls.append(hits / len(truth))
        ndcgs.append(ndcg_at_k(topk, truth, k))

    return {
        "K": k,
        "UsersEvaluated": int(len(users)),
        "Precision@K": float(np.mean(precisions)),
        "Recall@K": float(np.mean(recalls)),
        "NDCG@K": float(np.mean(ndcgs)),
    }

res5  = evaluate_lightgcn(k=5, max_users=2000)
res10 = evaluate_lightgcn(k=10, max_users=2000)

print(res5)
print(res10)


{'K': 5, 'UsersEvaluated': 2000, 'Precision@K': 0.16410000000000002, 'Recall@K': 0.41025, 'NDCG@K': 0.339576711945589}
{'K': 10, 'UsersEvaluated': 2000, 'Precision@K': 0.11685000000000002, 'Recall@K': 0.58425, 'NDCG@K': 0.4088938867730908}


In [11]:
# ============================================================
# Part 8 — Show LightGCN recommendations as actual joke text
# ============================================================

@torch.no_grad()
def recommend_lightgcn_for_user(raw_user_id: int, k: int = 5):
    """Top-k recommendations for one raw user_id, returned with joke text."""
    if raw_user_id not in user2idx:
        raise ValueError("raw_user_id not in filtered/train set (try a different user).")

    u = user2idx[raw_user_id]

    model.eval()
    users_final, items_final = model.propagate(A_norm)

    # Score all items for this user
    scores = torch.matmul(users_final[u].unsqueeze(0), items_final.T).squeeze(0)

    # Filter training items (don’t recommend already-liked training jokes)
    seen = train_user_items[int(u)]
    if len(seen) > 0:
        scores[list(seen)] = -1e9

    topk_idx = torch.topk(scores, k=k).indices.cpu().numpy().tolist()

    results = []
    for it_idx in topk_idx:
        joke_id = int(idx2item[it_idx])  # back to 1..100 joke_id
        score = float(scores[it_idx].cpu().item())
        text = jokes_df.loc[jokes_df["joke_id"] == joke_id, "joke_text"].iloc[0]
        results.append((joke_id, score, text))

    return results

# Use a user you know is in the test split
example_raw_user = int(test_pos["user_id"].iloc[0])
print("Example raw user_id:", example_raw_user)

recs = recommend_lightgcn_for_user(example_raw_user, k=5)
for rank, (jid, score, text) in enumerate(recs, start=1):
    print(f"\n#{rank} — Joke {jid} | score={score:.4f}\n{text[:500]}...")


Example raw user_id: 0

#1 — Joke 32 | score=0.3470
Joke 32
A man arrives at the gates of heaven. St. Peter asks, "Religion?"
The man says, "Methodist." St. Peter looks down his list, and says,
"Go to room 24, but be very quiet as you pass room 8."
Another man arrives at the gates of heaven. "Religion?"
"Baptist." "Go to room 18, but be very quiet as you pass room 8."
A third man arrives at the gates. "Religion?" "Jewish."
"Go to room 11, but be very quiet as you pass room 8."
The man says, "I can understand there being different rooms for differ...

#2 — Joke 61 | score=0.3296
A Joke
 During a recent publicity outing, Hillary sneaked off to visit a
fortune teller of some local repute. In a dark and hazy room, peering
into a crystal ball, the mystic delivered grave news.
 "There's no easy way to say this, so I'll just be blunt: Prepare
yourself to be a widow. Your husband will die a violent and horrible
death this year."
 Visibly shaken, Hillary stared at the woman's lined face, then a

In [12]:
# ============================================================
# Part 9 — Compare TF-IDF vs LightGCN (table)
# ============================================================

tfidf_5  = {"Precision@K": 0.0605, "Recall@K": 0.15125, "NDCG@K": 0.10619627997583679}
tfidf_10 = {"Precision@K": 0.0543, "Recall@K": 0.2715,  "NDCG@K": 0.15336266038360685}

lightgcn_5  = {"Precision@K": res5["Precision@K"],  "Recall@K": res5["Recall@K"],  "NDCG@K": res5["NDCG@K"]}
lightgcn_10 = {"Precision@K": res10["Precision@K"], "Recall@K": res10["Recall@K"], "NDCG@K": res10["NDCG@K"]}

summary = pd.DataFrame([
    {"Model": "TF-IDF",   "K": 5,  **tfidf_5},
    {"Model": "TF-IDF",   "K": 10, **tfidf_10},
    {"Model": "LightGCN", "K": 5,  **lightgcn_5},
    {"Model": "LightGCN", "K": 10, **lightgcn_10},
])

summary


,Model,K,Precision@K,Recall@K,NDCG@K
0,TF-IDF,5,0.06050,0.15125,0.106196
1,TF-IDF,10,0.05430,0.27150,0.153363
2,LightGCN,5,0.16410,0.41025,0.339577
3,LightGCN,10,0.11685,0.58425,0.408894
